# Likninger og nullpunkter

```{admonition} Læringsutbytte
Etter å ha arbeidet med denne delen av emnet, skal du kunne:

1. formulere en likning som et nullpunktsproblem $f(x)=0$
2. forklare og implementere halveringsmetoden og Newtons metode
3. bruke toleranse og maks antall iterasjoner til å kontrollere en numerisk beregning
4. vurdere styrker, svakheter og konvergens for ulike nullpunktsmetoder
5. bruke ferdige nullpunktsløsere i SciPy på kjemiske problemer
```

Mange kjemiske problemer ender med en likning vi må løse. Noen kan løses analytisk, men ofte blir uttrykkene så sammensatte at en numerisk løsning er mer praktisk.

Hovedideen er enkel: En likning

$$g(x)=h(x)$$

kan skrives som

$$f(x)=g(x)-h(x)=0.$$

Et **nullpunkt** til en funksjon er en $x$-verdi der funksjonsverdien er 0. Å løse likningen $g(x)=h(x)$ er derfor det samme som å finne nullpunktet til $f(x)=g(x)-h(x)$. Da har vi formulert likningen som et **nullpunktsproblem**.


## Et kjemisk eksempel: pH i en svak syre

Vi bruker en 0,010 M løsning av eddiksyre som eksempel. For en enprotisk svak syre med total konsentrasjon $C$ kan vi kombinere massebalansen og syrekonstanten og skrive konsentrasjonen av den korresponderende basen som

$$[\mathrm{A^-}]=C\frac{K_a}{[\mathrm{H_3O^+}]+K_a}.$$

Ladningsbalansen er

$$[\mathrm{H_3O^+}]=[\mathrm{A^-}]+[\mathrm{OH^-}],$$

og vannets ionprodukt gir

$$[\mathrm{OH^-}]=\frac{K_w}{[\mathrm{H_3O^+}]}.$$

Hvis vi setter $h=[\mathrm{H_3O^+}]$, kan hele problemet samles i én funksjon:

$$f(h)=h-C\frac{K_a}{h+K_a}-\frac{K_w}{h}.$$

pH-en finnes når ladningsbalansen er oppfylt, altså når $f(h)=0$.

```{admonition} Hvorfor er dette nyttig?
:class: note
Vi trenger ikke isolere $h$ algebraisk. Hvis vi kan beregne $f(h)$, kan vi lete numerisk etter verdien av $h$ som gjør funksjonen lik null.
```


In [1]:
import numpy as np
import matplotlib.pyplot as plt

C = 0.010
Ka = 1.75e-5
Kw = 1.0e-14

def ladningsbalanse(h):
    A_minus = C * Ka / (h + Ka)
    OH = Kw / h
    return h - A_minus - OH

h = np.logspace(-7, -2, 500)
plt.semilogx(h, ladningsbalanse(h))
plt.axhline(0)
plt.xlabel(r"$[\mathrm{H_3O^+}]$ (mol/L)")
plt.ylabel("Ladningsbalanse")
plt.show()


Grafen viser omtrent hvor nullpunktet ligger. Det er ofte lurt å **visualisere problemet før vi bruker en numerisk metode**. Da kan vi blant annet se om det finnes flere nullpunkter, og velge et fornuftig startintervall.

## Første idé: let etter et fortegnsskifte

Hvis $f(x_i)$ og $f(x_{i+1})$ har motsatt fortegn, og funksjonen er kontinuerlig mellom punktene, må det ligge minst ett nullpunkt i intervallet. Et enkelt søk kan derfor gå gjennom en rekke punkter og stoppe når fortegnet skifter.

Dette er ikke den mest effektive metoden, men ideen leder direkte til halveringsmetoden.


In [2]:
x = np.logspace(-7, -2, 1000)
y = ladningsbalanse(x)

for i in range(len(x) - 1):
    if y[i] * y[i + 1] < 0:
        a = x[i]
        b = x[i + 1]
        break

print("Nullpunktet ligger mellom", a, "og", b, "mol/L")
print("Et første estimat er midtpunktet:", (a + b)/2)


Nullpunktet ligger mellom 0.00040607720257003656 og 0.0004107840889965647 mol/L
Et første estimat er midtpunktet: 0.00040843064578330063


## Halveringsmetoden

Halveringsmetoden starter med et intervall $[a,b]$ der $f(a)$ og $f(b)$ har motsatt fortegn. Vi finner midtpunktet

$$m=\frac{a+b}{2}$$

og beholder den halvdelen av intervallet som fortsatt inneholder et fortegnsskifte. Prosessen gjentas til intervallet eller funksjonsverdien er liten nok.

Metoden er ikke alltid den raskeste, men den er svært robust når vi har et kontinuerlig problem og et gyldig startintervall.

```{admonition} Toleranse
:class: note
I numeriske beregninger bør vi vanligvis ikke teste `f(x) == 0`. Vi stopper heller når $|f(x)|$ er mindre enn en valgt toleranse.
```


In [3]:
def halveringsmetoden(f, a, b, tol=1e-10, maks_iterasjoner=100):
    if f(a) * f(b) > 0:
        raise ValueError("f(a) og f(b) må ha motsatt fortegn.")

    for i in range(maks_iterasjoner):
        m = (a + b) / 2

        if abs(f(m)) < tol:
            return m, i + 1

        if f(a) * f(m) < 0:
            b = m
        else:
            a = m

    raise RuntimeError("Metoden konvergerte ikke innen maks antall iterasjoner.")

h_null, antall = halveringsmetoden(ladningsbalanse, 1e-7, 1e-2)
pH = -np.log10(h_null)

print(f"[H3O+] = {h_null:.6e} mol/L")
print(f"pH = {pH:.3f}")
print("Iterasjoner:", antall)


[H3O+] = 4.096715e-04 mol/L
pH = 3.388
Iterasjoner: 27


### Prøv selv

Fullfør halveringsmetoden i editoren og bruk den til å finne pH i den svake syra.

<iframe src="../../basthon/?from=examples/numeriske_likninger_halvering.py" width="100%" height="600" frameborder="0" title="Prøv selv: halveringsmetoden" loading="lazy" allowfullscreen></iframe>


## Newtons metode

Halveringsmetoden bruker bare funksjonsverdier. Newtons metode bruker i tillegg den deriverte.

Studentene kjenner notasjonen $f'(x)$ fra før. I neste kapittel skal vi også bruke notasjonen $\frac{df}{dx}$; begge beskriver den deriverte av $f$ med hensyn på $x$.

Tangenten i punktet $x_n$ kan skrives

$$y=f(x_n)+f'(x_n)(x-x_n).$$

Hvis vi setter $y=0$ og løser med hensyn på $x$, får vi neste estimat:

$$x_{n+1}=x_n-\frac{f(x_n)}{f'(x_n)}.$$

Dette gjentas til $f(x_n)$ er tilstrekkelig nær null.


In [4]:
def newtons_metode(f, f_derivert, x0, tol=1e-10, maks_iterasjoner=50):
    x = x0

    for i in range(maks_iterasjoner):
        dfdx = f_derivert(x)
        if abs(dfdx) < 1e-14:
            raise RuntimeError("Den deriverte er for nær null.")

        x_ny = x - f(x) / dfdx

        if abs(f(x_ny)) < tol:
            return x_ny, i + 1

        x = x_ny

    raise RuntimeError("Metoden konvergerte ikke innen maks antall iterasjoner.")

def f(x):
    return x**2 - 2

def f_derivert(x):
    return 2*x

rot, antall = newtons_metode(f, f_derivert, x0=1.0)
print("Rot:", rot)
print("Iterasjoner:", antall)


Rot: 1.4142135623746899
Iterasjoner: 4


### Når Newton ikke oppfører seg pent

Newtons metode konvergerer ofte svært raskt, men den er mer følsom for startgjetningen. Den kan også få problemer hvis $f'(x)$ blir null eller svært liten.

For funksjonen

$$f(x)=x^3-2x+2$$

vil startgjetningen $x_0=0$ gi $x_1=1$, mens $x_1=1$ sender oss tilbake til $x_2=0$. Metoden havner altså i en syklus i stedet for å finne nullpunktet.


In [5]:
def f_problem(x):
    return x**3 - 2*x + 2

def df_problem(x):
    return 3*x**2 - 2

x = 0.0
for i in range(6):
    print(i, x)
    x = x - f_problem(x) / df_problem(x)


0 0.0
1 1.0
2 0.0
3 1.0
4 0.0
5 1.0


```{admonition} Underveisoppgave
:class: tip
Prøv andre startverdier. Hvilke startverdier fører til et nullpunkt, og hvilke gir problemer? Hva forteller dette om forskjellen mellom halveringsmetoden og Newtons metode?
```

## Ferdige løsere i SciPy

Når vi har forstått prinsippet, er det vanlig å bruke testede algoritmer fra numeriske biblioteker. `scipy.optimize.root_scalar` samler flere metoder for én-dimensjonale nullpunktsproblemer.


In [6]:
from scipy.optimize import root_scalar

bisect_resultat = root_scalar(ladningsbalanse, bracket=[1e-7, 1e-2], method="bisect")

def d_ladningsbalanse(h):
    return 1 + C*Ka/(h + Ka)**2 + Kw/h**2

newton_resultat = root_scalar(ladningsbalanse, x0=4e-4, fprime=d_ladningsbalanse, method="newton")

print("Halvering:")
print("  konvergert:", bisect_resultat.converged)
print("  iterasjoner:", bisect_resultat.iterations)
print("  pH:", -np.log10(bisect_resultat.root))

print("\nNewton:")
print("  konvergert:", newton_resultat.converged)
print("  iterasjoner:", newton_resultat.iterations)
print("  pH:", -np.log10(newton_resultat.root))


Halvering:
  konvergert: True
  iterasjoner: 33
  pH: 3.387564221805658

Newton:
  konvergert: True
  iterasjoner: 3
  pH: 3.3875642210277825


## Hvilken metode skal vi velge?

| Situasjon | Et naturlig valg |
|---|---|
| Vi kjenner et intervall med fortegnsskifte | Halvering eller en annen bracket-metode |
| Vi har en god startverdi og kjenner den deriverte | Newton |
| Vi vil ha en robust ferdig løser | `root_scalar` med et passende metodevalg |
| Det kan finnes flere nullpunkter | Plott eller skann området først |

Det viktigste er ikke bare å få et tall, men å kunne kontrollere at tallet faktisk løser det kjemiske problemet.

```{admonition} Numerisk arbeidsflyt
:class: important
1. Formuler kjemien som $f(x)=0$.
2. Undersøk funksjonen og velg et fornuftig søkeområde.
3. Velg metode, toleranse og eventuelt startgjett.
4. Kontroller at metoden konvergerte.
5. Sett løsningen tilbake i modellen og vurder om den er kjemisk rimelig.
```

## Kort oppsummering

- Likninger kan formuleres som nullpunktsproblemer.
- Halveringsmetoden er robust når vi har et fortegnsskifte.
- Newtons metode kan være rask, men er mer følsom for startgjetning og den deriverte.
- Toleranse og maks antall iterasjoner gjør beregningen kontrollerbar.
- SciPy gir ferdige løsere, men vi bør fortsatt forstå hva slags problem vi gir dem.


## Oppgaver

```{admonition} Oppgave 1 – pH i en svak syre
:class: tip
Bruk `halveringsmetoden` til å finne pH i 0,0250 M eddiksyre med $K_a=1.75\cdot10^{-5}$. Sammenlikn med tilnærmingen $[\mathrm{H_3O^+}]\approx\sqrt{K_aC}$. Hvor stor er forskjellen?
```

```{admonition} Oppgave 2 – velg metode
:class: tip
Du skal løse tre problemer:

1. En funksjon har et kjent fortegnsskifte mellom 2 og 3, men den deriverte er vanskelig å beregne.
2. Du kjenner en god startverdi og både $f(x)$ og $f'(x)$ er enkle å beregne.
3. Du mistenker at funksjonen har tre nullpunkter i intervallet $[-5,5]$.

Velg en arbeidsmåte for hvert tilfelle og begrunn valget.
```

```{admonition} Oppgave 3 – flere nullpunkter
:class: tip
Finn alle løsningene til $x^5=5x^3+3$. Plott først nullpunktsfunksjonen. Bruk deretter halveringsmetoden eller `root_scalar` på passende delintervaller.
```

```{admonition} Oppgave 4 – Newton og startgjett
:class: tip
Undersøk $f(x)=x^3-2x+2$ med Newtons metode. Test minst fem ulike startverdier. Forklar hvorfor samme metode kan lykkes fra én startverdi og mislykkes fra en annen.
```

```{admonition} Oppgave 5 – kjemisk likevekt
:class: tip
For reaksjonen $\mathrm{A \rightleftharpoons B}$ starter vi med 1,00 M A og 0 M B. Ved likevekt er $[B]=x$ og $[A]=1-x$. La $K=3.5$ og formuler likningen $K=[B]/[A]$ som et nullpunktsproblem. Finn $x$ numerisk, og kontroller løsningen analytisk.
```

```{admonition} Oppgave 6 – temperatur der en prosess skifter spontanitet
:class: tip
Anta at $\Delta H=45.0$ kJ/mol og $\Delta S=125$ J/(mol K) er konstante i et temperaturintervall. Formuler $\Delta G(T)=\Delta H-T\Delta S=0$ som et nullpunktsproblem og finn temperaturen. Denne likningen er enkel å løse analytisk; bruk den derfor til å kontrollere den numeriske metoden.
```


## Videoer

````{tab-set}
```{tab-item} Halveringsmetoden
<iframe width="890" height="500" src="https://www.youtube.com/embed/Ut7hUwPrHwo" title="YouTube video player" frameborder="0" allowfullscreen></iframe>
```
```{tab-item} Newtons metode
<iframe width="890" height="500" src="https://www.youtube.com/embed/7gCA5Per73g" title="YouTube video player" frameborder="0" allowfullscreen></iframe>
```
````
